# Grid Points Sensitivity Analysis

Evaluates how the number of intake grid points and log RR points affects model accuracy.
Both parameters are varied simultaneously across 13 values: 3, 4, 5, 6, 8, 10, 15, 20, 30, 40, 50, 70, 100.

The `grid_100` scenario serves as the reference for computing errors.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import yaml

In [ ]:
# Configuration
PROJECT_ROOT = Path("..").resolve()
CONFIG_NAME = "grid_sensitivity"
RESULTS_DIR = PROJECT_ROOT / "results" / CONFIG_NAME
CACHE_DIR = Path("cache") / CONFIG_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Grid point values in the analysis
GRID_VALUES = [3, 4, 5, 6, 8, 10, 15, 20, 30, 40, 50, 70, 100]
REFERENCE_GRID = 100  # Reference scenario for error calculations

# Load scenarios from config
config_path = PROJECT_ROOT / "config" / f"{CONFIG_NAME}.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

scenarios_path = PROJECT_ROOT / config["scenario_defs"]
with open(scenarios_path) as f:
    scenarios_def = yaml.safe_load(f)

print(f"Defined scenarios: {len(scenarios_def)}")
print(f"Grid values: {GRID_VALUES}")

## Load Solved Networks

In [ ]:
def load_scenarios() -> dict[int, Path]:
    """Load all grid scenario network paths."""
    solved_dir = RESULTS_DIR / "solved"
    scenarios = {}

    for grid_n in GRID_VALUES:
        network_path = solved_dir / f"model_scen-grid_{grid_n}.nc"
        if network_path.exists():
            scenarios[grid_n] = network_path
        else:
            print(f"Warning: Missing scenario grid_{grid_n}")

    print(f"Found {len(scenarios)} of {len(GRID_VALUES)} scenarios")
    return scenarios


scenarios = load_scenarios()

## Extract Food Group Consumption by Country

In [ ]:
# Constants for unit conversion
GRAMS_PER_MEGATONNE = 1e12
DAYS_PER_YEAR = 365


def extract_food_group_consumption(
    network_path: Path,
) -> pd.DataFrame:
    """Extract food group consumption by country in g/person/day.

    Returns:
        DataFrame with countries as index and food groups as columns (g/person/day).
    """
    n = pypsa.Network(network_path)
    snapshot = n.snapshots[-1]

    # Get population by country from network metadata
    pop_meta = n.meta.get("population", {})
    country_pop = pop_meta.get("country", {})

    # Get store levels at the snapshot
    if snapshot in n.stores_t.e.index:
        store_levels = n.stores_t.e.loc[snapshot]
    else:
        store_levels = pd.Series(dtype=float)

    # Extract food group stores (carrier starts with 'group_')
    fg_stores = n.stores[n.stores["carrier"].str.startswith("group_")]

    # Build consumption DataFrame
    consumption_data = []
    for store_name in fg_stores.index:
        level_mt = store_levels.get(store_name, 0.0)
        country = fg_stores.at[store_name, "country"]
        carrier = fg_stores.at[store_name, "carrier"]
        food_group = carrier.replace("group_", "")

        if pd.isna(country) or country not in country_pop:
            continue

        pop = country_pop[country]
        if pop > 0:
            # Convert Mt/year to g/person/day
            g_per_person_day = (level_mt * GRAMS_PER_MEGATONNE) / (pop * DAYS_PER_YEAR)
            consumption_data.append(
                {
                    "country": country,
                    "food_group": food_group,
                    "g_per_day": g_per_person_day,
                }
            )

    df = pd.DataFrame(consumption_data)
    if df.empty:
        return pd.DataFrame()

    # Pivot to get countries as rows, food groups as columns
    return df.pivot(index="country", columns="food_group", values="g_per_day").fillna(0)

In [ ]:
def extract_all_consumption(scenarios: dict[int, Path]) -> dict[int, pd.DataFrame]:
    """Extract consumption data for all scenarios."""
    consumption_data = {}

    for grid_n, network_path in sorted(scenarios.items()):
        print(f"Extracting consumption for grid_{grid_n}...")
        consumption_data[grid_n] = extract_food_group_consumption(network_path)

    return consumption_data


consumption_by_scenario = extract_all_consumption(scenarios)

# Show sample data
if REFERENCE_GRID in consumption_by_scenario:
    ref_consumption = consumption_by_scenario[REFERENCE_GRID]
    print(f"\nReference (grid_{REFERENCE_GRID}) shape: {ref_consumption.shape}")
    print(f"Countries: {len(ref_consumption)}")
    print(f"Food groups: {list(ref_consumption.columns)}")
    print("\nGlobal mean consumption (g/person/day):")
    print(ref_consumption.mean().round(1))

## Extract Objective Values

In [ ]:
def extract_objective_value(network_path: Path) -> float:
    """Extract the objective function value from a solved network."""
    n = pypsa.Network(network_path)
    return n.objective


def extract_all_objectives(scenarios: dict[int, Path]) -> pd.Series:
    """Extract objective values for all scenarios."""
    objectives = {}

    for grid_n, network_path in sorted(scenarios.items()):
        print(f"Extracting objective for grid_{grid_n}...")
        objectives[grid_n] = extract_objective_value(network_path)

    return pd.Series(objectives, name="objective")


objectives = extract_all_objectives(scenarios)
print("\nObjective values:")
print(objectives)

## Compute Error Metrics

In [ ]:
def compute_consumption_errors(
    consumption_by_scenario: dict[int, pd.DataFrame],
    reference_grid: int = REFERENCE_GRID,
) -> pd.DataFrame:
    """Compute error metrics for consumption relative to reference.

    Metrics:
    - RMSE: Root Mean Square Error across all (country, food_group) pairs
    - MAE: Mean Absolute Error
    - MAPE: Mean Absolute Percentage Error (for values > threshold)
    - Max Error: Maximum absolute error
    """
    if reference_grid not in consumption_by_scenario:
        raise ValueError(f"Reference grid_{reference_grid} not found")

    ref = consumption_by_scenario[reference_grid]

    metrics = []
    for grid_n, df in sorted(consumption_by_scenario.items()):
        # Ensure same countries and food groups
        common_countries = ref.index.intersection(df.index)
        common_groups = ref.columns.intersection(df.columns)

        ref_values = ref.loc[common_countries, common_groups].values.flatten()
        test_values = df.loc[common_countries, common_groups].values.flatten()

        errors = test_values - ref_values
        abs_errors = np.abs(errors)

        # RMSE
        rmse = np.sqrt(np.mean(errors**2))

        # MAE
        mae = np.mean(abs_errors)

        # MAPE (only for values > 1 g/day to avoid division issues)
        mask = ref_values > 1.0
        if mask.sum() > 0:
            mape = np.mean(abs_errors[mask] / ref_values[mask]) * 100
        else:
            mape = np.nan

        # Max error
        max_error = np.max(abs_errors)

        metrics.append(
            {
                "grid_points": grid_n,
                "rmse_g_per_day": rmse,
                "mae_g_per_day": mae,
                "mape_percent": mape,
                "max_error_g_per_day": max_error,
            }
        )

    return pd.DataFrame(metrics).set_index("grid_points")


consumption_errors = compute_consumption_errors(consumption_by_scenario)
print("Consumption error metrics:")
print(consumption_errors.round(4))

In [ ]:
def compute_objective_errors(
    objectives: pd.Series,
    reference_grid: int = REFERENCE_GRID,
) -> pd.DataFrame:
    """Compute objective value errors relative to reference."""
    if reference_grid not in objectives.index:
        raise ValueError(f"Reference grid_{reference_grid} not found")

    ref_obj = objectives[reference_grid]

    metrics = []
    for grid_n in objectives.index:
        obj_val = objectives[grid_n]
        abs_diff = abs(obj_val - ref_obj)
        rel_diff = (obj_val - ref_obj) / ref_obj * 100 if ref_obj != 0 else np.nan

        metrics.append(
            {
                "grid_points": grid_n,
                "objective_bn_usd": obj_val / 1e9,  # Convert to billions
                "abs_diff_bn_usd": abs_diff / 1e9,
                "rel_diff_percent": rel_diff,
            }
        )

    return pd.DataFrame(metrics).set_index("grid_points")


objective_errors = compute_objective_errors(objectives)
print("Objective value errors:")
print(objective_errors.round(4))

## Visualizations

In [ ]:
# Combine all metrics for plotting
all_metrics = pd.merge(
    consumption_errors.reset_index(),
    objective_errors.reset_index(),
    on="grid_points",
)
all_metrics = all_metrics.set_index("grid_points")
print("Combined metrics:")
print(all_metrics)

In [ ]:
def plot_convergence(all_metrics: pd.DataFrame):
    """Create convergence plots showing how errors decrease with grid points."""
    # Exclude reference (which has 0 error)
    df = all_metrics[all_metrics.index != REFERENCE_GRID].copy()

    fig, axes = plt.subplots(2, 2, figsize=(10, 8))

    # Panel a: RMSE convergence
    ax = axes[0, 0]
    ax.loglog(
        df.index, df["rmse_g_per_day"], "o-", color="C0", linewidth=2, markersize=6
    )
    ax.set_xlabel("Number of grid points")
    ax.set_ylabel("RMSE [g/person/day]")
    ax.set_title("(a) Root Mean Square Error")
    ax.grid(True, alpha=0.3, which="both")
    ax.text(-0.1, 1.05, "a", transform=ax.transAxes, fontsize=12, fontweight="bold")

    # Panel b: MAE convergence
    ax = axes[0, 1]
    ax.loglog(
        df.index, df["mae_g_per_day"], "o-", color="C1", linewidth=2, markersize=6
    )
    ax.set_xlabel("Number of grid points")
    ax.set_ylabel("MAE [g/person/day]")
    ax.set_title("(b) Mean Absolute Error")
    ax.grid(True, alpha=0.3, which="both")
    ax.text(-0.1, 1.05, "b", transform=ax.transAxes, fontsize=12, fontweight="bold")

    # Panel c: MAPE convergence
    ax = axes[1, 0]
    ax.semilogx(
        df.index, df["mape_percent"], "o-", color="C2", linewidth=2, markersize=6
    )
    ax.set_xlabel("Number of grid points")
    ax.set_ylabel("MAPE [%]")
    ax.set_title("(c) Mean Absolute Percentage Error")
    ax.grid(True, alpha=0.3, which="both")
    ax.text(-0.1, 1.05, "c", transform=ax.transAxes, fontsize=12, fontweight="bold")

    # Panel d: Objective value convergence
    ax = axes[1, 1]
    ax.semilogx(
        df.index, df["rel_diff_percent"], "o-", color="C3", linewidth=2, markersize=6
    )
    ax.axhline(y=0, color="black", linestyle="--", alpha=0.5)
    ax.set_xlabel("Number of grid points")
    ax.set_ylabel("Objective difference [%]")
    ax.set_title("(d) Objective Value Relative Difference")
    ax.grid(True, alpha=0.3, which="both")
    ax.text(-0.1, 1.05, "d", transform=ax.transAxes, fontsize=12, fontweight="bold")

    plt.tight_layout()

    # Save figure
    output_dir = PROJECT_ROOT / "notebooks" / "figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        output_dir / "grid_sensitivity_convergence.pdf", dpi=300, bbox_inches="tight"
    )
    print(f"Saved to: {output_dir / 'grid_sensitivity_convergence.pdf'}")

    plt.show()


plot_convergence(all_metrics)

In [ ]:
def plot_combined_error_metrics(all_metrics: pd.DataFrame):
    """Create a combined plot showing multiple error metrics on log-log scale."""
    # Exclude reference
    df = all_metrics[all_metrics.index != REFERENCE_GRID].copy()

    fig, ax = plt.subplots(figsize=(8, 5))

    # Normalize metrics to 0-1 scale for comparison
    rmse_norm = df["rmse_g_per_day"] / df["rmse_g_per_day"].max()
    mae_norm = df["mae_g_per_day"] / df["mae_g_per_day"].max()
    mape_norm = df["mape_percent"] / df["mape_percent"].max()
    obj_norm = df["abs_diff_bn_usd"] / df["abs_diff_bn_usd"].max()

    ax.loglog(df.index, rmse_norm, "o-", label="RMSE (normalized)", markersize=6)
    ax.loglog(df.index, mae_norm, "s-", label="MAE (normalized)", markersize=6)
    ax.loglog(df.index, mape_norm, "^-", label="MAPE (normalized)", markersize=6)
    ax.loglog(
        df.index, obj_norm, "d-", label="Objective diff (normalized)", markersize=6
    )

    ax.set_xlabel("Number of grid points", fontsize=10)
    ax.set_ylabel("Normalized error (relative to max)", fontsize=10)
    ax.set_title("Grid Points Sensitivity: Error Convergence", fontsize=12)
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3, which="both")

    # Add reference lines for common convergence rates
    x = np.array(df.index)
    ax.loglog(x, 0.5 * (x[0] / x), "--", color="gray", alpha=0.5, label="O(1/n)")
    ax.loglog(x, 0.5 * (x[0] / x) ** 2, ":", color="gray", alpha=0.5, label="O(1/n^2)")

    plt.tight_layout()

    # Save figure
    output_dir = PROJECT_ROOT / "notebooks" / "figures"
    fig.savefig(
        output_dir / "grid_sensitivity_combined.pdf", dpi=300, bbox_inches="tight"
    )
    print(f"Saved to: {output_dir / 'grid_sensitivity_combined.pdf'}")

    plt.show()


plot_combined_error_metrics(all_metrics)

## Summary Table

In [ ]:
def create_summary_table(all_metrics: pd.DataFrame) -> pd.DataFrame:
    """Create a summary table of key metrics."""
    df = all_metrics.copy()

    # Rename columns for clarity
    df = df.rename(
        columns={
            "rmse_g_per_day": "RMSE [g/day]",
            "mae_g_per_day": "MAE [g/day]",
            "mape_percent": "MAPE [%]",
            "max_error_g_per_day": "Max Error [g/day]",
            "objective_bn_usd": "Objective [bn USD]",
            "abs_diff_bn_usd": "Obj Diff [bn USD]",
            "rel_diff_percent": "Obj Diff [%]",
        }
    )

    return df.round(4)


summary_table = create_summary_table(all_metrics)
print("Summary Table:")
print(summary_table.to_string())

In [ ]:
# Export summary table to CSV
summary_table.to_csv(CACHE_DIR / "summary_metrics.csv")
print(f"Saved summary table to: {CACHE_DIR / 'summary_metrics.csv'}")

## Key Findings

Summarize the key findings from the analysis:

1. **Convergence rate**: How quickly do errors decrease with increasing grid points?
2. **Practical threshold**: At what number of grid points do errors become negligible?
3. **Trade-off**: What is the accuracy vs computation time trade-off?

In [ ]:
# Find threshold where RMSE < 1 g/day (effectively negligible)
threshold_rmse = 1.0  # g/day
threshold_mape = 1.0  # %

df = all_metrics[all_metrics.index != REFERENCE_GRID]

rmse_threshold_met = df[df["rmse_g_per_day"] < threshold_rmse].index.min()
mape_threshold_met = df[df["mape_percent"] < threshold_mape].index.min()

print(f"Grid points needed for RMSE < {threshold_rmse} g/day: {rmse_threshold_met}")
print(f"Grid points needed for MAPE < {threshold_mape}%: {mape_threshold_met}")

# Estimate convergence rate from log-log slope
if len(df) > 2:
    log_n = np.log(df.index.values)
    log_rmse = np.log(df["rmse_g_per_day"].values + 1e-10)
    slope = np.polyfit(log_n, log_rmse, 1)[0]
    print(f"\nEstimated convergence rate: O(n^{slope:.2f})")